In [1]:
import sqlite3
import pandas as pd
import numpy as np
from tqdm import tqdm
import json

In [2]:
SQLITE_PATH = "dev_ai_articles_full.sqlite"


dev_articles = "articles"
DEV_COLUMN_CONFIG = {
    "id": "id",           # primary key column
    "title": "title",     # article title column
    "body": "body_text",       # article body/content column (set to None if not available)
}

hashnode_articles = "hashnode_articles"
HASHNODE_COLUMN_CONFIG = {
    "id": "id",           # primary key column
	"title": "title",     # article title column
	"body": "body",       # article body/content column (set to None if not available)
}

# Output table where results will be written
OUTPUT_TABLE = "article_classifications"

# How much body text to use (tokens are expensive; first 500 chars is usually enough)
BODY_PREVIEW_CHARS = 1500

# Batch size for zero-shot classification (lower if you run out of RAM)
CLASSIFICATION_BATCH_SIZE = 32

In [3]:
conn = sqlite3.connect(SQLITE_PATH)
cursor = conn.cursor()

In [4]:
id_col = DEV_COLUMN_CONFIG["id"]
title_col = DEV_COLUMN_CONFIG["title"]
body_col = DEV_COLUMN_CONFIG["body"]

query = f"SELECT {id_col}, {title_col}, {body_col} FROM {dev_articles}"

df = pd.read_sql_query(query, conn)


hashnode_df = pd.read_sql_query(f"SELECT {id_col}, {title_col}, {body_col} FROM {hashnode_articles}", conn)




In [5]:
df

,id,title,body_text
0,142249,The Distressed Code Review,title: The Distressed Code Review published: t...
1,198370,Avoid These Terrible Web Notifications Mistake...,This article is a brief introduction to some b...
2,385335,DNS Explained. Resolution,This is an article in the DNS Explained. serie...
3,423055,You don't need a library for state machines,title: You don't need a library for state mach...
4,441453,Data Engineering Series #3: Apache Airflow - t...,Why such attention towards Airflow ? Interest ...
...,...,...,...
3019,3363277,Hardware Hacking 101 Village - Post-Mortem,title: Hardware Hacking 101 Village Post Morte...
3020,3363288,FE/BE - Unite Them!,tl;dr; Teams should agree upon and understand ...
3021,3363939,Announcing the Colab MCP Server: Connect Any A...,When you’re prototyping locally with AI agents...
3022,3364128,I Built a Claude Code Agent That Doesn't Need ...,title: I Built a Claude Code Agent That Doesn'...


In [6]:
hashnode_df

,id,title,body_text
0,69c4b8a8efeaf33e6b3675be,MonALISA : A Distributed Monitoring Service Ar...,Adaptive Monitoring for Large Scale Grids: a S...
1,69c4bd952d879655ece508e5,The Sweet Spot of AI Orchestration: From AWS L...,The Sweet Spot of AI Orchestration: From AWS L...
2,69c4aaa5bbdb1bc33f100e25,"45 Claude Code Hooks for Code Quality, Securit...",45 Claude Code Hooks I Use to Automate Code Qu...
3,69c4a9faaed4ec64674bd0db,Claude Code Channels: Set Up a 24/7 AI Agent v...,Claude Code Channels Just Dropped — Here's How...
4,69c4a88eea32df99834d3704,I Built My Own OpenClaw Alternative With Claud...,I Built My Own OpenClaw Alternative With Claud...
...,...,...,...
22643,69a85632e55311e40f0eeb63,Building an AI-Powered Hospital Infection Inte...,A Case Study from CODE:AUTOMATA Ver 2.1 Hackat...
22644,69a8580ae55311e40f0fd641,You don't know your AI stack.,AI tooling has no supply chain. No audit trail...
22645,69a8589de55311e40f1023c3,How AI Is Changing Software Architecture Planning,Most AI conversations in software focus on cod...
22646,69a6cd5c75d7a0f10015ca0d,How to Run and Customize LLMs Locally with Ollama,In the long history of technological innovatio...


In [7]:
df_all = pd.concat([df, hashnode_df], ignore_index=True)

In [8]:
df_all

,id,title,body_text
0,142249,The Distressed Code Review,title: The Distressed Code Review published: t...
1,198370,Avoid These Terrible Web Notifications Mistake...,This article is a brief introduction to some b...
2,385335,DNS Explained. Resolution,This is an article in the DNS Explained. serie...
3,423055,You don't need a library for state machines,title: You don't need a library for state mach...
4,441453,Data Engineering Series #3: Apache Airflow - t...,Why such attention towards Airflow ? Interest ...
...,...,...,...
25667,69a85632e55311e40f0eeb63,Building an AI-Powered Hospital Infection Inte...,A Case Study from CODE:AUTOMATA Ver 2.1 Hackat...
25668,69a8580ae55311e40f0fd641,You don't know your AI stack.,AI tooling has no supply chain. No audit trail...
25669,69a8589de55311e40f1023c3,How AI Is Changing Software Architecture Planning,Most AI conversations in software focus on cod...
25670,69a6cd5c75d7a0f10015ca0d,How to Run and Customize LLMs Locally with Ollama,In the long history of technological innovatio...


Now that dfs are combined, we can start clustering

In [9]:
df_all["combined_text"] = (
	df_all[title_col].fillna("") + " " +
	df_all[body_col].fillna("").str[:BODY_PREVIEW_CHARS]
).str.strip()

In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # ~80MB, fast, good quality

texts = df_all["combined_text"].tolist()
embeddings = model.encode(
	texts,
	batch_size=64,
	show_progress_bar=True,
	convert_to_numpy=True,
)
print(f"✅ Embeddings shape: {embeddings.shape}")


embeddings

c:\Users\abbhi\anaconda3\envs\CIS4930\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Batches: 100%|██████████| 402/402 [28:11<00:00,  4.21s/it]


✅ Embeddings shape: (25672, 384)


array([[-0.03045662, -0.02122168, -0.02453277, ..., -0.03061731,
         0.06307959,  0.06322043],
       [-0.03914866, -0.03178781,  0.001767  , ..., -0.01439707,
        -0.03717563,  0.06547669],
       [-0.01395674, -0.05800174,  0.05732978, ...,  0.06991705,
        -0.03069576,  0.03327543],
       ...,
       [ 0.0204586 ,  0.05308988,  0.05026504, ...,  0.00267067,
        -0.02624441, -0.02031039],
       [ 0.02364533, -0.07605472,  0.0279224 , ...,  0.04681316,
        -0.05939626, -0.01434456],
       [-0.04372468, -0.0524913 , -0.02987417, ..., -0.06269559,
         0.07843874, -0.06325853]], shape=(25672, 384), dtype=float32)

In [11]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA

n_clusters = 10


# Reduce dims first for speed
pca = PCA(n_components=50, random_state=42)
reduced = pca.fit_transform(embeddings)

km = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, n_init=3)
labels = km.fit_predict(reduced)

unique, counts = np.unique(labels, return_counts=True)
for u, c in zip(unique, counts):
	print(f"   Cluster {u:2d}: {c:,} articles")


labels

   Cluster  0: 3,212 articles
   Cluster  1: 2,580 articles
   Cluster  2: 3,939 articles
   Cluster  3: 2,989 articles
   Cluster  4: 2,031 articles
   Cluster  5: 1,992 articles
   Cluster  6: 2,437 articles
   Cluster  7: 2,368 articles
   Cluster  8: 2,409 articles
   Cluster  9: 1,715 articles


array([2, 2, 2, ..., 3, 0, 4], shape=(25672,), dtype=int32)

In [12]:
n_samples = 10
df_all_copy = df_all.copy()
df_all_copy["cluster"] = labels
for cluster_id in sorted(df_all_copy["cluster"].unique()):
	samples = df_all_copy[df_all_copy["cluster"] == cluster_id]["combined_text"].head(n_samples).tolist()
	print(f"\n  ── Cluster {cluster_id} ──")
	for s in samples:
		print(f"    • {s[:120]}")



  ── Cluster 0 ──
    • Basic Setup For a Machine Learning Project With Python In this tutorial, we present the process for configuring a basic 
    • Build a Personalized Smart Alarm Clock with Python Hello readers, you often would have seen or build alarm clocks using 
    • 40 "Free Forever" Tools for Developers Hey everyone, I am Hrishikesh, co founder of Flexiple, an exclusive freelance net
    • Build a CLI with Node.js Command line utilities are the most basic and beautiful apps ever created, the apps that starte
    • Integrating reCAPTCHA with Next.js In this post on integrating reCAPTCHA with Next.js, we will be looking at what is a C
    • 10 Best Python IDEs and Code Editors to use in 2021 While you can write Python with just a simple text editor, using a c
    • Build a Chuck Norris Discord Bot in Python [Discord.Py] for beginners This is a quick guide on how to create a Chuck Nor
    • A Brief Tour of the Unity Editor title: A Brief Tour of the Unity Editor published: tr

In [18]:
# Keep only relevant clusters since zero-shot classification is expensive
relevant_clusters = [3, 8]

# Filtered df with only relevant clusters
df_filtered = df_all_copy[df_all_copy["cluster"].isin(relevant_clusters)].copy()

print(f"Original articles: {len(df_all_copy)}")
print(f"Filtered articles: {len(df_filtered)}")
print(f"Removed: {len(df_all_copy) - len(df_filtered)}")


Original articles: 25672
Filtered articles: 5398
Removed: 20274


In [21]:
df_filtered

,id,title,body_text,combined_text,cluster
279,585948,Diamond Hands and the Mechanics of a Short Squ...,title: Diamond Hands and the Mechanics of a Sh...,Diamond Hands and the Mechanics of a Short Squ...,8
396,604165,12 Important Software Testing Trends for 2021 ...,Software testing is making many moves. From AI...,12 Important Software Testing Trends for 2021 ...,3
508,628993,"Another 8th of March, yet we are not still there!",Where are we about equality in tech in 2021? A...,"Another 8th of March, yet we are not still the...",3
588,645109,Top Automation Testing Trends To Look Out In 2021,"Back in the old days, software testing was jus...",Top Automation Testing Trends To Look Out In 2...,3
683,971871,How Cryptocurrency Works Explained Visually,"In 2008, when Lehman Brothers Holdings Inc. fi...",How Cryptocurrency Works Explained Visually In...,8
...,...,...,...,...,...
25653,69ae116a86766ac3a663b05f,The Copper Ceiling: AI's High-Voltage Collisio...,"When we interact with artificial intelligence,...",The Copper Ceiling: AI's High-Voltage Collisio...,3
25655,699d48cad00f02a90785119e,Physical AI: Why the Grid’s Operating System S...,As someone working in Developer Relations in t...,Physical AI: Why the Grid’s Operating System S...,3
25665,69a889213b896f7ad6749182,I Thought AI Made Me Faster. My Metrics Disagr...,"Friday, 4:47 PM. A PR lands in the repo with a...",I Thought AI Made Me Faster. My Metrics Disagr...,3
25668,69a8580ae55311e40f0fd641,You don't know your AI stack.,AI tooling has no supply chain. No audit trail...,You don't know your AI stack. AI tooling has n...,3


Zero-shot classification

In [ ]:
CANDIDATE_LABELS_PRIMARY = [
    "a coding tutorial, technical guide, or software implementation walkthrough with little discussion of societal implications",
    "an opinion piece, analysis, or commentary on AI's consequences for society, law, environment, or ethics",
]


CANDIDATE_LABELS_SECONDARY = [
    "discussion of the environmental impact or energy consumption of AI",
    "discussion of legal, regulatory, or policy issues related to AI",
    "discussion of ethical concerns, bias, or fairness in AI systems",
    "discussion of AI's impact on jobs, employment, or the economy",
    "discussion of AI in geopolitics, international competition, or national security",
    "discussion of how AI is portrayed in media or public opinion",
    "discussion of AI safety, alignment, or existential risks",
]


In [34]:
def classify_batch(pipe, texts, candidate_labels, threshold=0.5):
    results = pipe(texts, candidate_labels, multi_label=True)
    if isinstance(results, dict):
        results = [results]
    
    output = []
    for r in results:
        labels_scores = [
            (label, score)
            for label, score in zip(r["labels"], r["scores"])
            if score >= threshold
        ]
        output.append(labels_scores)
    
    return output

In [ ]:
from transformers import pipeline

pipe = pipeline(
	"zero-shot-classification",
	model="facebook/bart-large-mnli",
	device=-1,  # CPU; change to 0 if you have a CUDA GPU
)

texts = df_filtered["combined_text"].tolist()
primary_labels = []
primary_scores = []


for i in tqdm(range(0, 30, CLASSIFICATION_BATCH_SIZE)):
    batch = texts[i : i + CLASSIFICATION_BATCH_SIZE]
    results = classify_batch(pipe, batch, CANDIDATE_LABELS_PRIMARY, threshold=0.0)

    for r in results:
        # r is like: [("technical ...", 0.88), ("non-technical ...", 0.12)]
        if not r:
            primary_labels.append("UNSURE")
            primary_scores.append(0.0)
            continue
        if max(r[0][1], r[1][1]) < 0.7:  # If the top label is below threshold or less confident than second, mark as UNSURE
            primary_labels.append("UNSURE")
            primary_scores.append(0.0)
            continue
        top_label, top_score = r[0]
        is_technical = "technical" in top_label.lower()
        primary_labels.append("TECHNICAL" if is_technical else "NON_TECHNICAL")
        primary_scores.append(float(top_score))

df_filtered["primary_label"][:30] = primary_labels[:30]
df_filtered["primary_score"][:30] = primary_scores[:30]

Device set to use cpu
100%|██████████| 1/1 [01:41<00:00, 101.62s/it]


[0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.7055606245994568,
 0.0,
 0.71224445104599,
 0.0,
 0.0,
 0.0,
 0.8673710823059082,
 0.0,
 0.0,
 0.0,
 0.7544775009155273]

In [43]:
# # Sub-classify only the non-technical ones
# non_tech_mask = df_filtered["primary_label"] == "NON_TECHNICAL"
# non_tech_texts = df_filtered.loc[non_tech_mask, "combined_text"].tolist()
# non_tech_indices = df_filtered.index[non_tech_mask].tolist()

# print(f"\n   Found {len(non_tech_texts):,} non-technical articles")
# print("   Step 2/2: Sub-categorizing non-technical articles...")

# secondary_labels = [""] * len(df_filtered)
# secondary_scores = [0.0] * len(df_filtered)

# for i, (idx, text) in enumerate(tqdm(zip(non_tech_indices, non_tech_texts))):
# 	result = classify_batch(pipe, [text], CANDIDATE_LABELS_SECONDARY)[0]
# 	secondary_labels[idx] = result["label"]
# 	secondary_scores[idx] = result["score"]

# df_filtered["secondary_label"] = secondary_labels
# df_filtered["secondary_score"] = secondary_scores

# print("✅ Classification complete")

In [26]:
import sqlite3

conn = sqlite3.connect("dev_ai_articles_full.sqlite")
df_filtered.to_sql("topic labeled articles", conn, if_exists="replace", index=False)
conn.close()

In [50]:
# Sub-classify only the non-technical ones
non_tech_mask = df_filtered["primary_label"] != "TECHNICAL"
non_tech_texts = df_filtered.loc[non_tech_mask, "combined_text"].tolist()
non_tech_indices = df_filtered.index[non_tech_mask].tolist()

print(f"\n   Found {len(non_tech_texts):,} non-technical articles")
print("   Step 2/2: Sub-categorizing non-technical articles...")

secondary_labels = [""] * len(df_filtered)
secondary_scores = [0.0] * len(df_filtered)

for i in tqdm(range(0, len(non_tech_texts), CLASSIFICATION_BATCH_SIZE)):
    batch_texts = non_tech_texts[i : i + CLASSIFICATION_BATCH_SIZE]
    batch_indices = non_tech_indices[i : i + CLASSIFICATION_BATCH_SIZE]

    results = classify_batch(
        pipe,
        batch_texts,
        CANDIDATE_LABELS_SECONDARY,
        threshold=0.0
    )

    for idx, r in zip(batch_indices, results):
        if not r:
            secondary_labels[idx] = "UNSURE"
            secondary_scores[idx] = 0.0
            continue

        top_label, top_score = r[0]
        secondary_labels[idx] = top_label
        secondary_scores[idx] = float(top_score)

df_filtered["secondary_label"] = secondary_labels
df_filtered["secondary_score"] = secondary_scores

print("✅ Classification complete")


   Found 27 non-technical articles
   Step 2/2: Sub-categorizing non-technical articles...


100%|██████████| 1/1 [04:32<00:00, 272.82s/it]

✅ Classification complete


In [59]:
secondary_rows = df_filtered[df_filtered["secondary_score"] > 0]

secondary_rows

,id,title,body_text,combined_text,cluster,primary_label,primary_score,secondary_label,secondary_score
3832,69b689b7e2e3d79e70701b0c,Why I Help People Install OpenClaw for Free,A friend asked me: Why do you help others inst...,Why I Help People Install OpenClaw for Free A ...,3,TECHNICAL,0.429741,discussion of how AI is portrayed in media or ...,0.104395
4840,69a3fccca7428b958d932943,My journey from knowing nothing about AI to la...,Three months ago I decided to take a leap of f...,My journey from knowing nothing about AI to la...,3,TECHNICAL,0.764546,"discussion of AI safety, alignment, or existen...",0.700552
5488,69941af7f25a1798f7f17234,The demand for great\nsoftware testers is risi...,Here's what actually happens when companies go...,The demand for great\nsoftware testers is risi...,3,TECHNICAL,0.812198,"discussion of ethical concerns, bias, or fairn...",0.512709
5941,698d2e9379fa9a9349915c3f,AI Statistics: 40+ Key Stats for 2026,1. Home › 2. Digital Marketing › 3. Artificial...,AI Statistics: 40+ Key Stats for 2026 1. Home ...,3,TECHNICAL,0.081523,"discussion of AI safety, alignment, or existen...",0.419531
6257,69898756f0fd5ecc1e770fda,Will ENS Token Recover Above $9 After Abandoni...,Will ENS Token Recover Above $9 After Abandoni...,Will ENS Token Recover Above $9 After Abandoni...,8,TECHNICAL,0.686379,"discussion of AI safety, alignment, or existen...",0.014862
6261,69897effe60b684a9c35d4ce,You Feel 30% Faster with AI (But Tests Show Yo...,You're using AI to code. You feel faster. Way ...,You Feel 30% Faster with AI (But Tests Show Yo...,3,TECHNICAL,0.495077,discussion of how AI is portrayed in media or ...,0.226759
6338,6988c74012513d340449ae63,Robinhood Earnings February 2026: Will HOOD St...,Robinhood Earnings February 2026: Will HOOD St...,Robinhood Earnings February 2026: Will HOOD St...,8,TECHNICAL,0.570299,discussion of how AI is portrayed in media or ...,0.124911
6363,69884ede8840a29134bc86ad,Will Trump's US-India Interim Trade Agreement ...,Will Trump's US India Interim Trade Agreement ...,Will Trump's US-India Interim Trade Agreement ...,8,TECHNICAL,0.424062,"discussion of ethical concerns, bias, or fairn...",0.245351
6617,6987200ea6071d824cb44c99,"Wolves vs Chelsea February 7, 2026: 77% Probab...","Wolves vs Chelsea February 7, 2026: 77% Probab...","Wolves vs Chelsea February 7, 2026: 77% Probab...",8,TECHNICAL,0.556602,"discussion of AI's impact on jobs, employment,...",0.300913
6642,698700cc8a3260e124629677,"US x Iran Diplomatic Meeting by February 13, 2...","US x Iran Diplomatic Meeting by February 13, 2...","US x Iran Diplomatic Meeting by February 13, 2...",8,TECHNICAL,0.573541,discussion of how AI is portrayed in media or ...,0.404944


In [32]:
high_confidence = df_filtered[df_filtered["primary_score"] > 0.7]

In [33]:
len(high_confidence)

1265

In [37]:
primary_scores

[0.19885492324829102,
 0.5728352069854736,
 0.46954262256622314,
 0.5771845579147339,
 0.020828502252697945,
 0.2536979913711548,
 0.26505202054977417,
 0.2804679572582245,
 0.11881227046251297,
 0.4245702028274536,
 0.4228259325027466,
 0.2017759382724762,
 0.591781497001648,
 0.5281444191932678,
 0.11178062856197357,
 0.1781216859817505,
 0.6177603602409363,
 0.6041975617408752,
 0.3047257661819458,
 0.4220218360424042,
 0.09906215965747833,
 0.7055606245994568,
 0.0025816811248660088,
 0.71224445104599,
 0.38881155848503113,
 0.06214163824915886,
 0.48241525888442993,
 0.8673710823059082,
 0.45976459980010986,
 0.1215277686715126,
 0.5673112869262695,
 0.7544775009155273]

In [47]:
df_filtered


,id,title,body_text,combined_text,cluster,primary_label,primary_score,secondary_label,secondary_score
279,585948,Diamond Hands and the Mechanics of a Short Squ...,title: Diamond Hands and the Mechanics of a Sh...,Diamond Hands and the Mechanics of a Short Squ...,8,TECHNICAL,0.198855,,0.0
396,604165,12 Important Software Testing Trends for 2021 ...,Software testing is making many moves. From AI...,12 Important Software Testing Trends for 2021 ...,3,TECHNICAL,0.572835,,0.0
508,628993,"Another 8th of March, yet we are not still there!",Where are we about equality in tech in 2021? A...,"Another 8th of March, yet we are not still the...",3,TECHNICAL,0.469543,,0.0
588,645109,Top Automation Testing Trends To Look Out In 2021,"Back in the old days, software testing was jus...",Top Automation Testing Trends To Look Out In 2...,3,TECHNICAL,0.577185,,0.0
683,971871,How Cryptocurrency Works Explained Visually,"In 2008, when Lehman Brothers Holdings Inc. fi...",How Cryptocurrency Works Explained Visually In...,8,TECHNICAL,0.020829,,0.0
...,...,...,...,...,...,...,...,...,...
25653,69ae116a86766ac3a663b05f,The Copper Ceiling: AI's High-Voltage Collisio...,"When we interact with artificial intelligence,...",The Copper Ceiling: AI's High-Voltage Collisio...,3,TECHNICAL,0.291620,,0.0
25655,699d48cad00f02a90785119e,Physical AI: Why the Grid’s Operating System S...,As someone working in Developer Relations in t...,Physical AI: Why the Grid’s Operating System S...,3,TECHNICAL,0.263547,,0.0
25665,69a889213b896f7ad6749182,I Thought AI Made Me Faster. My Metrics Disagr...,"Friday, 4:47 PM. A PR lands in the repo with a...",I Thought AI Made Me Faster. My Metrics Disagr...,3,TECHNICAL,0.331377,,0.0
25668,69a8580ae55311e40f0fd641,You don't know your AI stack.,AI tooling has no supply chain. No audit trail...,You don't know your AI stack. AI tooling has n...,3,TECHNICAL,0.603208,,0.0


In [49]:
df_filtered.loc[df_filtered.index[:30], "primary_label"] = primary_labels[:30]
df_filtered.loc[df_filtered.index[:30], "primary_score"] = primary_scores[:30]

In [60]:
!pip install gensim

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   - -------------------------------------- 0.8/24.4 MB 3.7 MB/s eta 0:00:07
   -- ------------------------------------- 1.6/24.4 MB 4.0 MB/s eta 0:00:06
   --- ------------------------------------ 2.4/24.4 MB 3.9 MB/s eta 0:00:06
   ----- ---------------------------------- 3.1/24.4 MB 3.9 MB/s eta 0:00:06
   ------ --------------------------------- 3.9/24.4 MB 3.9 MB/s eta 0:00:06
   -------- ------------------------------- 5.0/24.4 MB 3.9 MB/s eta 0:00:05
   --------- ------------------------------ 5.8/24.4 MB 3.9 MB/s eta 0:00:05
   ---------- ----------------------------- 6.6/24.4 MB 3.9 MB/s eta 0:00:05
   ------------ --------------------------- 7.3/24.4 MB 3.9 MB/s eta 0:00:05
   ------------- -------------------------- 8.1/24.4 MB 3.9 MB/s eta 0:00:05
   -------------- ------------------------- 8.9/24.4 MB 3.9 MB/s eta 0:00:04
   --------------- ------------------------ 9.7/24.4 MB 3.9 MB/s eta 0:00:04
   ---

In [61]:
import gensim.corpora as corpora
from gensim.models import LdaModel

# Tokenize
texts_tokenized = [text.lower().split() for text in texts]

# Dictionary & corpus
dictionary = corpora.Dictionary(texts_tokenized)
corpus = [dictionary.doc2bow(text) for text in texts_tokenized]

# LDA
lda_model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=5, passes=10)
lda_model.print_topics()

[(0,
  '0.054*"the" + 0.018*"in" + 0.017*"and" + 0.016*"of" + 0.016*"a" + 0.014*"to" + 0.011*"market" + 0.011*"for" + 0.010*"will" + 0.010*"with"'),
 (1,
  '0.040*"the" + 0.027*"and" + 0.025*"ai" + 0.021*"to" + 0.021*"a" + 0.020*"of" + 0.015*"in" + 0.013*"is" + 0.010*"that" + 0.008*"for"'),
 (2,
  '0.038*"the" + 0.014*"as" + 0.013*"federal" + 0.012*"fed" + 0.012*"in" + 0.012*"of" + 0.012*"reserve" + 0.011*"to" + 0.010*"will" + 0.010*"trump"'),
 (3,
  '0.074*"|" + 0.029*"the" + 0.015*"in" + 0.013*"a" + 0.012*"market" + 0.010*"to" + 0.010*"and" + 0.010*"of" + 0.009*"bitcoin" + 0.008*"will"'),
 (4,
  '0.054*"the" + 0.017*"a" + 0.017*"in" + 0.011*"to" + 0.010*"with" + 0.008*"this" + 0.008*"and" + 0.008*"for" + 0.008*"on" + 0.007*"super"')]